# APUSH LEQ Grader SLM — v2 gated training and evaluation

This Colab notebook builds and trains the audited **data-improvement v2** artifacts, then evaluates the tuned model on the selected 53-row College Board case file at `artifacts/data/eval_cb_cases.jsonl`.

Current local status: 87 independently accepted realistic cases, 9 pending review records, and no release-ready v2 training artifact yet.

## 1. Confirm the GPU

In [47]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Attach a GPU runtime before continuing."

Fri Jul 10 22:18:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone an explicit repository revision

Set `GIT_REF` to the commit containing the v2 scripts and artifacts. A commit SHA is strongly preferred over `main` for reproducibility.

In [48]:
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/aryanjverma/slm.git"
REPO_DIR = Path("/content/slm")
GIT_REF = os.environ.get("SLM_GIT_REF", "main")  # Prefer a committed SHA.

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "--prune", "--tags"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", GIT_REF], check=True)
remote_ref = f"refs/remotes/origin/{GIT_REF}"
remote_branch_exists = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--verify", remote_ref],
    capture_output=True,
).returncode == 0
if remote_branch_exists:
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "merge", "--ff-only", f"origin/{GIT_REF}"],
        check=True,
    )
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Repository commit:", commit)

required = [
    Path("scripts/build_v2_artifacts.py"),
    Path("scripts/review_synthetic_v2.py"),
    Path("scripts/run_v2_checkpoints.py"),
    Path("scripts/eval_hf_model.py"),
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, f"The selected revision does not contain the v2 pipeline: {missing}"

Repository commit: 646547fe6cc36fcbc251ae6ad9b5c824baf1dad2


## 3. Install dependencies

In [49]:
!pip install -q -e ".[train]" sentencepiece tiktoken

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Building editable for apush-frq-grader-slm (pyproject.toml) ... done


## 4. Optional Hugging Face token

In [ ]:
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        print("HF_TOKEN loaded from Colab secrets.")
    else:
        print("No HF_TOKEN configured; downloads will be anonymous.")
except Exception as exc:
    print("No Colab HF_TOKEN available:", exc)

## 5. Configure the v2 run

The canonical training input is `artifacts/data/v2/train_chat_v2.jsonl`, produced by the audited builder. The similarly named root-level 800-row file is a legacy template oversample and is never selected here.

In [50]:
import json, math, hashlib, sys, time
from collections import Counter

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_NAME = "apush_frq_grader_v2"
MODEL_DIR = Path("artifacts/models/apush-frq-grader-v2")
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
assert DRIVE_ROOT.is_dir(), "A mounted Google Drive is required for resumable evaluation."
EVAL_DIR = DRIVE_ROOT / "slm_evaluation/apush-frq-grader-v2"
V2_DIR = Path("artifacts/data/v2")

UNREVIEWED = Path("artifacts/data/train_realistic_v2_unreviewed.jsonl")
REVIEW_LOG = Path("artifacts/reviews/synthetic_v2.jsonl")
REVIEWED = Path("artifacts/data/train_realistic_v2_reviewed.jsonl")
LEGACY_ADVERSARIAL = Path("artifacts/data/train_cases.jsonl")
LEGACY_TEMPLATE_V2 = Path("artifacts/data/train_chat_v2.jsonl")
TRAIN_CHAT = V2_DIR / "train_chat_v2.jsonl"
MANIFEST = V2_DIR / "dataset_manifest_v2.json"

CB_EVAL = Path("artifacts/data/eval_cb_cases.jsonl")
CB_AUDIT = Path("artifacts/audits/cb_golden_v2.json")

ADVERSARIAL_RATIO = 0.15
TARGET_CAP = 1200
CHECKPOINT_COUNTS = [200, 500, 1200]
EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 8
RUN_BASE_EVAL = False  # Opt in only after the deterministic refresh succeeds.
RUN_CHECKPOINT_SWEEP = False
MAX_NEW_TOKENS = 512
FORCE_REBUILD = False

EVAL_DIR.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 6. Inventory available data

In [51]:
def read_jsonl(path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

def row_count(path):
    return len(read_jsonl(path))

print(f"Legacy v1 chat rows:       {row_count(Path('artifacts/data/train_chat.jsonl'))}")
print(f"Legacy template-v2 rows:  {row_count(LEGACY_TEMPLATE_V2)} (not canonical v2)")
print(f"Accepted realistic rows:  {row_count(UNREVIEWED)} (not train-ready until review)")
print(f"Completed reviewed rows:  {row_count(REVIEWED)}")
print(f"Canonical v2 chat rows:   {row_count(TRAIN_CHAT)}")
print(f"Selected CB eval rows:    {row_count(CB_EVAL)}")
if CB_AUDIT.exists():
    cb_audit = json.loads(CB_AUDIT.read_text(encoding="utf-8"))
    print(f"CB audit warning: {len(cb_audit.get('accepted_ids', []))}/{cb_audit.get('total', 0)} verified; evaluation will still use all selected rows.")

if LEGACY_TEMPLATE_V2.exists():
    legacy = read_jsonl(LEGACY_TEMPLATE_V2)
    print(f"Legacy template-v2 unique IDs: {len({row['id'] for row in legacy})}")

Legacy v1 chat rows:       1000
Legacy template-v2 rows:  800 (not canonical v2)
Accepted realistic rows:  87 (not train-ready until review)
Completed reviewed rows:  87
Canonical v2 chat rows:   102
Selected CB eval rows:    53
CB audit warning: 0/53 verified; evaluation will still use all selected rows.
Legacy template-v2 unique IDs: 800


## 7. Enforce the human-review gate

A review record is accepted only when it has a reviewer and all three verification flags are true. Do not set these fields automatically. If any sampled case fails review, correct or replace it before building the dataset.

In [ ]:
reviews = read_jsonl(REVIEW_LOG)

def review_accepted(row):
    return bool(
        row.get("reviewer")
        and row.get("scores_verified")
        and row.get("feedback_verified")
        and row.get("historical_accuracy_verified")
    )

accepted_reviews = [row for row in reviews if review_accepted(row)]
pending_reviews = [row for row in reviews if not review_accepted(row)]
print(f"Review queue: {len(reviews)} total, {len(accepted_reviews)} accepted, {len(pending_reviews)} pending/failed")
if pending_reviews:
    print("STOP: complete artifacts/reviews/synthetic_v2.jsonl locally, commit the reviewed log, then rerun.")
    print("Pending case IDs:", [row.get("case_id") for row in pending_reviews])


## 8. Apply reviews and build the immutable v2 mix

With 87 realistic rows and a 15% adversarial share, the largest current mix is 102 rows. This is a pilot retrain, not a 200-row checkpoint.

In [ ]:
assert reviews, f"Missing review log: {REVIEW_LOG}"
assert not pending_reviews, "Human-review gate is incomplete; refusing to build training data."

subprocess.run([
    sys.executable, "scripts/review_synthetic_v2.py",
    "--input", str(UNREVIEWED),
    "--review-log", str(REVIEW_LOG),
    "--output", str(REVIEWED),
], check=True)

realistic_rows = row_count(REVIEWED)
adversarial_rows = sum(
    row.get("failure_type") in {"grade_inflation_request", "prompt_injection"}
    for row in read_jsonl(LEGACY_ADVERSARIAL)
)
possible = [
    count for count in range(1, TARGET_CAP + 1)
    if count - round(count * ADVERSARIAL_RATIO) <= realistic_rows
    and round(count * ADVERSARIAL_RATIO) <= adversarial_rows
]
assert possible, "No eligible v2 training mix can be assembled."
target_count = max(possible)
print(f"Building {target_count} rows from {realistic_rows} realistic and {adversarial_rows} eligible adversarial rows.")

command = [
    sys.executable, "scripts/build_v2_artifacts.py",
    "--realistic", str(REVIEWED),
    "--adversarial", str(LEGACY_ADVERSARIAL),
    "--output-dir", str(V2_DIR),
    "--target-count", str(target_count),
    "--adversarial-ratio", str(ADVERSARIAL_RATIO),
]
if MANIFEST.exists() and TRAIN_CHAT.exists() and not FORCE_REBUILD:
    print(f"Using existing immutable artifact: {MANIFEST}")
else:
    if FORCE_REBUILD:
        command.append("--force")
    subprocess.run(command, check=True)

## 9. Verify manifest hashes and release gates

In [ ]:
manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
audit = manifest["audit"]
assert not audit.get("global_reasons"), audit.get("global_reasons")
assert not audit.get("rejected"), audit.get("rejected")
assert audit.get("human_review_rate", 0) >= 0.10

for record in manifest["artifacts"]:
    path = Path(record["path"].replace("\\", "/"))
    if not path.exists():
        raise FileNotFoundError(f"Manifest artifact is missing: {path}")
    payload = path.read_bytes()
    actual_sha256 = hashlib.sha256(payload).hexdigest()
    if len(payload) != record["bytes"] or actual_sha256 != record["sha256"]:
        raise RuntimeError(
            f"Manifest mismatch for {path}. Rebuild the v2 artifacts with the current revision; "
            "do not train from modified bytes."
        )
    if path.suffix == ".jsonl":
        assert row_count(path) == record["rows"], path

train_rows = row_count(TRAIN_CHAT)
assert train_rows > 0
assert TRAIN_CHAT.parent == V2_DIR, "Refusing the legacy root-level train_chat_v2.jsonl."
print(f"Manifest verified: {train_rows} canonical v2 SFT rows; human review rate={audit['human_review_rate']:.2%}")

## 10. Materialize available 200/500/1,200 checkpoints

Below 200 rows, the checkpoint plan is intentionally empty and the next cell runs only a clearly labeled pilot model.

In [52]:
checkpoint_command = [
    sys.executable, "scripts/run_v2_checkpoints.py",
    "--data", str(TRAIN_CHAT),
    "--counts", *map(str, CHECKPOINT_COUNTS),
    "--model", BASE_MODEL,
    "--cb-eval", str(CB_EVAL),
    "--output-root", "artifacts/checkpoints/v2",
    "--eval-output-root", str(EVAL_DIR / "checkpoints"),
]
if RUN_CHECKPOINT_SWEEP:
    checkpoint_command.append("--execute")
subprocess.run(checkpoint_command, check=True)
plan = json.loads(Path("artifacts/checkpoints/v2/checkpoint_plan.json").read_text(encoding="utf-8"))
print(f"Materialized {len(plan.get('runs', []))} formal checkpoints from {train_rows} rows.")

Materialized 0 formal checkpoints from 102 rows.


## 11. Train the current v2 artifact

Steps are scaled to approximately three epochs with an effective batch size of 16, with a 30-step minimum for the small pilot.

In [ ]:
effective_batch = BATCH_SIZE * GRAD_ACCUM
max_steps = max(30, math.ceil(train_rows / effective_batch) * EPOCHS)
print(f"Training {train_rows} rows for {max_steps} steps; output={MODEL_DIR}")
subprocess.run([
    sys.executable, "scripts/train_qlora.py",
    "--model", BASE_MODEL,
    "--data", str(TRAIN_CHAT),
    "--output", str(MODEL_DIR),
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum", str(GRAD_ACCUM),
    "--max-steps", str(max_steps),
    "--seed", "13",
], check=True)

## 12. Optional base-model comparison on the selected CB cases

The base-model run is optional because it requires a separate model download and GPU inference. When enabled, it uses the same selected 53-row College Board evaluation file as the tuned model.

In [53]:
if RUN_BASE_EVAL:
    subprocess.run([
        sys.executable, "scripts/eval_hf_model.py",
        "--model", BASE_MODEL,
        "--model-name", "qwen_base_v2_cb_eval",
        "--eval-path", str(CB_EVAL),
        "--output-dir", str(EVAL_DIR),
        "--real-eval", "--resume",
        "--max-new-tokens", str(MAX_NEW_TOKENS),
    ], check=True)

In [55]:
import shutil
from pathlib import Path

local_config = MODEL_DIR / "adapter_config.json"
if local_config.exists():
    print(f"Using adapter already present at {MODEL_DIR}")
else:
    drive_model_root = Path("/content/drive/MyDrive/slm_models/apush-frq-grader-v2")
    candidates = sorted(
        drive_model_root.rglob("adapter_config.json"),
        key=lambda path: (len(path.parts), path.as_posix()),
    ) if drive_model_root.exists() else []
    if not candidates:
        raise FileNotFoundError(
            f"No saved v2 adapter found below {drive_model_root}. "
            "Run the training cell first, or save the trained adapter to Drive."
        )
    src = candidates[0].parent
    assert (src / "adapter_model.safetensors").exists(), f"Incomplete adapter backup: {src}"
    shutil.copytree(src, MODEL_DIR, dirs_exist_ok=True)
    print(f"Restored adapter from {src} to {MODEL_DIR}")

Using adapter already present at artifacts/models/apush-frq-grader-v2


## 13. Evaluate the selected College Board cases

The tuned model evaluates all 53 rows in `artifacts/data/eval_cb_cases.jsonl`. Results are written directly to mounted Drive and resume by case ID after a disconnect. The existing audit remains visible as a data-quality warning but does not block this explicitly selected evaluation.

In [56]:
def run_hf_eval(label, path):
    assert path.exists(), f"Missing selected evaluation file: {path}"
    assert row_count(path) == 53, f"Expected 53 selected CB cases, found {row_count(path)}"
    case_count = row_count(path)
    started = time.monotonic()
    print(f"START {label}: {case_count} cases from {path}", flush=True)
    command = [
        sys.executable, "-u", "scripts/eval_hf_model.py",
        "--model", str(MODEL_DIR),
        "--model-name", f"{MODEL_NAME}_{label}",
        "--eval-path", str(path),
        "--output-dir", str(EVAL_DIR),
        "--log-every", "10",
        "--max-new-tokens", str(MAX_NEW_TOKENS),
        "--real-eval", "--resume",
    ]
    subprocess.run(command, check=True)
    print(f"DONE {label}: elapsed={(time.monotonic() - started) / 60:.1f}m", flush=True)

run_hf_eval("cb_eval", CB_EVAL)

START cb_eval: 53 cases from artifacts/data/eval_cb_cases.jsonl
DONE cb_eval: elapsed=18.3m


In [ ]:
import json
from pathlib import Path

RESULTS = Path(
    "/content/drive/MyDrive/slm_evaluation/apush-frq-grader-v2/"
    "apush_frq_grader_v2_cb_eval_real_results.jsonl"
)
CASES = Path("artifacts/data/eval_cb_cases.jsonl")

def read_jsonl(path):
    if not path.exists():
        return []
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            pass
    return rows

cases = read_jsonl(CASES)
results = read_jsonl(RESULTS)

done_ids = {row["case_id"] for row in results}
missing = [row["id"] for row in cases if row["id"] not in done_ids]

print(f"Progress: {len(done_ids)}/{len(cases)} ({len(done_ids)/len(cases):.1%})")
print(f"Missing:  {len(missing)}")
print(f"Last completed: {results[-1]['case_id'] if results else 'none'}")

if missing:
    print("Next case:", missing[0])
else:
    print("Evaluation complete.")

## 14. Aggregate summaries for the selected CB evaluation

In [57]:
import hashlib, json, math, os, subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/slm")
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
from google.colab import drive
drive.mount("/content/drive")
EVAL_DIR = Path("/content/drive/MyDrive/slm_evaluation/apush-frq-grader-v2")
MODEL_DIR = Path("artifacts/models/apush-frq-grader-v2")
MANIFEST = Path("artifacts/data/v2/dataset_manifest_v2.json")
TRAIN_CHAT = Path("artifacts/data/v2/train_chat_v2.jsonl")
CB_EVAL = Path("artifacts/data/eval_cb_cases.jsonl")
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

def read_jsonl(path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
train_rows = len(read_jsonl(TRAIN_CHAT))
max_steps = max(30, math.ceil(train_rows / 16) * 3)
assert MANIFEST.exists(), f"Missing manifest: {MANIFEST}"
assert CB_EVAL.exists() and len(read_jsonl(CB_EVAL)) == 53

summary_rows = []
for path in sorted(EVAL_DIR.glob("*_summary.jsonl")):
    for row in read_jsonl(path):
        summary_rows.append({"file": path.name, **row})

if not summary_rows:
    print("No evaluation summaries found.")
else:
    preferred = [
        "model_name", "count", "structured_output_valid_rate",
        "exact_match_rate", "within_one_rate", "total_exact_match_rate",
        "total_within_one_rate", "total_mae", "qwk",
        "evidence_grounding_rate", "no_hallucination_rate", "robustness_mean",
    ]
    for row in summary_rows:
        print("\n", row["file"])
        print({key: row.get(key) for key in preferred if key in row})

report = {
    "repository_commit": commit,
    "base_model": BASE_MODEL,
    "model_path": str(MODEL_DIR),
    "manifest_path": str(MANIFEST),
    "manifest_sha256": hashlib.sha256(MANIFEST.read_bytes()).hexdigest(),
    "training_rows": train_rows,
    "max_steps": max_steps,
    "evaluation_path": str(CB_EVAL),
    "evaluation_sha256": hashlib.sha256(CB_EVAL.read_bytes()).hexdigest(),
    "evaluation_rows": 53,
    "max_new_tokens": 512,
    "summaries": summary_rows,
}
report_path = EVAL_DIR / "v2_run_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Run report:", report_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

 apush_frq_grader_v2_cb_eval_real_summary.jsonl
{'model_name': 'apush_frq_grader_v2_cb_eval', 'count': 53, 'structured_output_valid_rate': 0.3585, 'exact_match_rate': 0.4009, 'within_one_rate': 0.7123, 'total_exact_match_rate': 0.1509, 'total_within_one_rate': 0.3208, 'total_mae': 3.2075, 'qwk': 0.0531, 'evidence_grounding_rate': 0.7547}
Run report: /content/drive/MyDrive/slm_evaluation/apush-frq-grader-v2/v2_run_report.json


## 15. Optional: save adapter, manifest, and aggregate report to Drive

Keep per-case evaluation outputs private when copying the adapter and aggregate report.

In [58]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
eval_dir = Path("/content/drive/MyDrive/slm_evaluation/apush-frq-grader-v2")
print(sorted(path.name for path in eval_dir.glob("*cb_eval*")))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['apush_frq_grader_v2_cb_eval_real_by_rubric.jsonl', 'apush_frq_grader_v2_cb_eval_real_results.jsonl', 'apush_frq_grader_v2_cb_eval_real_results_diagnostics.json', 'apush_frq_grader_v2_cb_eval_real_results_dimensions.json', 'apush_frq_grader_v2_cb_eval_real_summary.jsonl']


In [59]:
from google.colab import drive
drive.mount("/content/drive")
backup = Path("/content/drive/MyDrive/slm_models/apush-frq-grader-v2")
backup.mkdir(parents=True, exist_ok=True)
subprocess.run(["cp", "-r", str(MODEL_DIR), str(backup / "adapter")], check=True)
subprocess.run(["cp", str(MANIFEST), str(backup / MANIFEST.name)], check=True)
subprocess.run(["cp", str(EVAL_DIR / "v2_run_report.json"), str(backup / "v2_run_report.json")], check=True)
print("Saved:", backup)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved: /content/drive/MyDrive/slm_models/apush-frq-grader-v2
